# Image to 3D — quality evaluation harness

Runs your own images through **TripoSR**, **Hunyuan3D 2.0** and **TRELLIS** so you can judge
whether open-source image-to-3D is good enough for furniture / interior assets before you
commit to an architecture.

### Before running anything
`Runtime` -> `Change runtime type` -> **T4 GPU** (free tier is enough).

### What to expect on a free T4 (16 GB)

| Section | Install | Per image | Fits T4? |
|---|---|---|---|
| 3. TripoSR | ~2 min | ~10 sec | Yes, easily |
| 4. Hunyuan3D 2.0 | ~10 min | 2-5 min | Yes (tight with texture) |
| 5. TRELLIS | ~25 min | 1-2 min | Yes, but fragile install |

Run sections top to bottom. **Sections 3/4/5 are independent** — if one fails, the others still work.
Do section 3 first so you have a result in your hands within five minutes.

> ### Read this before uploading images
> These are **single-object** reconstructors. A whole-room photo produces garbage.
> Crop each image to **one piece of furniture**, roughly centred, mostly filling the frame.
> Section 2 removes the background for you, which materially improves all three models.

---
## 1. Check the GPU

This also picks the right attention backend. **flash-attn requires Ampere (compute 8.0+)** —
a free-tier T4 is compute 7.5, so it must use `xformers` instead. Getting this wrong is the
single most common reason TRELLIS fails on Colab.

In [ ]:
import os, sys, subprocess, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

if not torch.cuda.is_available():
    raise SystemExit("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")

name  = torch.cuda.get_device_name(0)
cc    = torch.cuda.get_device_capability(0)
vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU            : {name}")
print(f"Compute cap    : {cc[0]}.{cc[1]}")
print(f"VRAM           : {vram:.1f} GB")
print(f"torch / cuda   : {torch.__version__} / {torch.version.cuda}")

# --- capability-driven config used by every section below -------------------
AMPERE_PLUS = cc[0] >= 8
DTYPE       = torch.bfloat16 if AMPERE_PLUS else torch.float16   # Turing (T4) has no usable bf16
ATTN        = "flash-attn"   if AMPERE_PLUS else "xformers"

os.environ["ATTN_BACKEND"]      = ATTN
os.environ["SPCONV_ALGO"]       = "native"          # avoids slow spconv autotune on first run
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print()
print(f"-> attention backend : {ATTN}")
print(f"-> dtype             : {DTYPE}")
print()
if vram < 14:
    print("WARNING: under 14 GB. TripoSR will work; Hunyuan3D texture and TRELLIS will likely OOM.")
elif vram < 20:
    print("NOTE: 16 GB class GPU. Hunyuan3D shape+texture sits right at the limit —")
    print("      section 4 frees the shape model before painting to make it fit.")
else:
    print("Plenty of VRAM. Everything below should run comfortably.")

---
## 2. Upload and prepare your images

Upload 2-4 cropped furniture shots. Anything you upload gets background-removed and
padded to a square, which is what all three models expect.

In [ ]:
import os, glob, shutil
from pathlib import Path

WORK = Path("/content/eval3d")
RAW, IN, OUT = WORK/"raw", WORK/"in", WORK/"out"
for d in (RAW, IN, OUT): d.mkdir(parents=True, exist_ok=True)

from google.colab import files
print("Select 2-4 cropped images (one object each). Cancel to use a built-in sample.")
uploaded = files.upload()

for fn in uploaded:
    shutil.move(fn, RAW/fn)

if not any(RAW.iterdir()):
    print("\nNothing uploaded — fetching a sample chair image instead.")
    os.system(f"wget -q -O {RAW}/sample_chair.png "
              "https://raw.githubusercontent.com/VAST-AI-Research/TripoSR/main/examples/chair.png")

print("\nRaw inputs:")
for p in sorted(RAW.iterdir()): print("  ", p.name)

In [ ]:
# Background removal + square padding. Skipping this step noticeably degrades all three models.
!pip install -q rembg onnxruntime pillow

from rembg import remove, new_session
from PIL import Image

session = new_session("u2net")

def prep(src: Path, dst: Path, size: int = 1024, margin: float = 0.9):
    im = Image.open(src).convert("RGBA")
    im = remove(im, session=session)                  # -> transparent background

    bbox = im.getchannel("A").getbbox()               # tight-crop to the object
    if bbox: im = im.crop(bbox)

    scale = (size * margin) / max(im.size)
    im = im.resize((max(1,int(im.width*scale)), max(1,int(im.height*scale))), Image.LANCZOS)

    canvas = Image.new("RGBA", (size, size), (0, 0, 0, 0))
    canvas.paste(im, ((size-im.width)//2, (size-im.height)//2), im)
    canvas.save(dst)
    return canvas

IMAGES = []
for src in sorted(RAW.iterdir()):
    if src.suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp"}: continue
    dst = IN / (src.stem + ".png")
    prep(src, dst)
    IMAGES.append(dst)
    print("prepared:", dst.name)

# quick visual check
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(IMAGES), figsize=(4*len(IMAGES), 4))
for ax, p in zip([axes] if len(IMAGES) == 1 else axes, IMAGES):
    ax.imshow(Image.open(p)); ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

print(f"\n{len(IMAGES)} image(s) ready in {IN}")

---
## 3. TripoSR — fast baseline

Weakest of the three, but installs in ~2 minutes and runs in ~10 seconds per image.
Treat its output as the **floor**: if TripoSR is already good enough for your use case,
you do not need a 24 GB GPU at all.

In [ ]:
%cd /content
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git 2>/dev/null || echo "already cloned"
%cd /content/TripoSR
!pip install -q --upgrade setuptools
!pip install -q -r requirements.txt

# torchmcubes builds a CUDA extension and is the usual failure point.
import importlib, sys
try:
    importlib.import_module("torchmcubes"); print("\ntorchmcubes OK")
except Exception as e:
    print("\ntorchmcubes missing/broken -> rebuilding from source:", e)
    !pip uninstall -y -q torchmcubes
    !pip install -q git+https://github.com/tatsy/torchmcubes.git

In [ ]:
%cd /content/TripoSR
!python run.py --help      # confirm the flag names for this checkout before we rely on them

In [ ]:
%cd /content/TripoSR
import subprocess, shlex
from pathlib import Path

WORK = Path("/content/eval3d"); IN = WORK/"in"; OUT = WORK/"out"
IMAGES = sorted(IN.glob("*.png"))
dst = OUT/"triposr"; dst.mkdir(parents=True, exist_ok=True)

cmd = ["python", "run.py", *[str(p) for p in IMAGES],
       "--output-dir", str(dst),
       "--bake-texture", "--texture-resolution", "2048",
       "--model-save-format", "glb",
       "--chunk-size", "8192"]        # lower to 4096/2048 if you ever hit OOM

print(" ".join(shlex.quote(c) for c in cmd), "\n")
subprocess.run(cmd, check=False)

print("\nProduced:")
for p in sorted(dst.rglob("*.glb")): print("  ", p, f"{p.stat().st_size/1e6:.1f} MB")

---
## 4. Hunyuan3D 2.0 — the quality benchmark

This is the one that decides your architecture. Shape generation is 6 GB; shape **and**
texture is ~16 GB, exactly a T4's capacity, so the run cell below explicitly frees the
shape pipeline before loading the paint pipeline.

**Why 2.0 and not 2.1:** Hunyuan3D 2.1 needs 21 GB for texture / 29 GB for both — it
cannot fit on a free T4. If you upgrade to an A100 later, switch the repo IDs to 2.1.

Install is ~10 minutes; the two `setup.py install` steps compile CUDA extensions.

In [ ]:
%cd /content
!git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git 2>/dev/null || echo "already cloned"
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .

# Two CUDA extensions for the texture stage. Skip-able if you only want geometry.
!cd hy3dgen/texgen/custom_rasterizer      && python setup.py install 2>&1 | tail -3
!cd hy3dgen/texgen/differentiable_renderer && python setup.py install 2>&1 | tail -3
print("\nHunyuan3D install finished.")

In [ ]:
# ---- Stage 1: geometry only (~6 GB) ---------------------------------------
%cd /content/Hunyuan3D-2
import torch, gc
from pathlib import Path

WORK = Path("/content/eval3d"); IN = WORK/"in"; OUT = WORK/"out"
IMAGES = sorted(IN.glob("*.png"))
dst = OUT/"hunyuan3d"; dst.mkdir(parents=True, exist_ok=True)

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

# 'tencent/Hunyuan3D-2mini' (0.6B) is the ~5 GB fallback if this OOMs.
REPO = "tencent/Hunyuan3D-2"

shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(REPO)
try:
    shape.enable_flashvdm()      # big speedup, present in recent checkouts only
except Exception as e:
    print("flashvdm unavailable:", e)

meshes = {}
for img in IMAGES:
    print(f"\n--- shape: {img.name}")
    mesh = shape(image=str(img))[0]
    out = dst / f"{img.stem}_shape.glb"
    mesh.export(str(out))
    meshes[img.stem] = out
    print("   ->", out.name, f"{out.stat().st_size/1e6:.1f} MB")

# free ~6 GB so the paint pipeline has room on a 16 GB card
del shape; gc.collect(); torch.cuda.empty_cache()
print(f"\nVRAM in use after cleanup: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# ---- Stage 2: texture (needs the freed VRAM from the cell above) -----------
%cd /content/Hunyuan3D-2
import torch, gc
from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2")

for stem, shape_path in meshes.items():
    img = IN / f"{stem}.png"
    print(f"\n--- texture: {stem}")
    try:
        import trimesh
        textured = paint(trimesh.load(str(shape_path)), image=str(img))
        out = dst / f"{stem}_textured.glb"
        textured.export(str(out))
        print("   ->", out.name, f"{out.stat().st_size/1e6:.1f} MB")
    except torch.cuda.OutOfMemoryError:
        print("   OOM — texture stage does not fit. Options: use a paid A100/L4 runtime,")
        print("   or evaluate geometry only and texture separately.")
        torch.cuda.empty_cache(); break

del paint; gc.collect(); torch.cuda.empty_cache()

---
## 5. TRELLIS — best quality, most fragile install

Optional. ~25 minutes to install because it compiles several CUDA kernels
(`diffoctreerast`, `nvdiffrast`, `spconv`, `mipgaussian`).

Two deviations from the official README, both required on Colab:
- **no `--new-env`** — the README creates a conda env that Colab's kernel cannot see.
- **no `--flash-attn`** on a T4 — Turing is compute 7.5 and flash-attn needs 8.0+. Section 1
  already set `ATTN_BACKEND=xformers` to cover this.

If this section fails, it does not affect your results from sections 3 and 4.

In [ ]:
%cd /content
!git clone -q --recurse-submodules https://github.com/microsoft/TRELLIS.git 2>/dev/null || echo "already cloned"
%cd /content/TRELLIS

import os
FLAGS = "--basic --xformers --spconv --mipgaussian --kaolin --nvdiffrast --diffoctreerast"
if os.environ.get("ATTN_BACKEND") == "flash-attn":
    FLAGS += " --flash-attn"          # Ampere+ only

print("setup.sh", FLAGS)
!. ./setup.sh {FLAGS}

In [ ]:
%cd /content/TRELLIS
import os
os.environ.setdefault("ATTN_BACKEND", "xformers")
os.environ.setdefault("SPCONV_ALGO", "native")

import torch, gc
from PIL import Image
from pathlib import Path
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

WORK = Path("/content/eval3d"); IN = WORK/"in"; OUT = WORK/"out"
dst = OUT/"trellis"; dst.mkdir(parents=True, exist_ok=True)

pipe = TrellisImageTo3DPipeline.from_pretrained("microsoft/TRELLIS-image-large")
pipe.cuda()

for img in sorted(IN.glob("*.png")):
    print(f"\n--- trellis: {img.name}")
    outputs = pipe.run(Image.open(img), seed=1)
    glb = postprocessing_utils.to_glb(
        outputs["gaussian"][0], outputs["mesh"][0],
        simplify=0.95, texture_size=1024,
    )
    out = dst / f"{img.stem}.glb"
    glb.export(str(out))
    print("   ->", out.name, f"{out.stat().st_size/1e6:.1f} MB")

del pipe; gc.collect(); torch.cuda.empty_cache()

---
## 6. Compare the results

Renders every GLB produced above in an interactive viewer, grouped by source image, so you
can spin them side by side. This is the cell that actually answers "is it good enough".

In [ ]:
import base64, html
from pathlib import Path
from IPython.display import HTML, display

OUT = Path("/content/eval3d/out")
MAX_MB = 25   # inlined as base64; larger files are listed but not previewed

found = sorted(OUT.rglob("*.glb"))
if not found:
    display(HTML("<b>No GLB files yet — run section 3, 4 or 5 first.</b>"))
else:
    cards = []
    for p in found:
        mb  = p.stat().st_size / 1e6
        tag = f"{p.parent.parent.name if p.parent.parent != OUT else p.parent.name}/{p.name}"
        if mb > MAX_MB:
            cards.append(f"<div class=c><b>{html.escape(tag)}</b>"
                         f"<p>{mb:.1f} MB — too large to preview inline, download it below.</p></div>")
            continue
        b64 = base64.b64encode(p.read_bytes()).decode()
        cards.append(
            f"<div class=c><b>{html.escape(tag)}</b><span>{mb:.1f} MB</span>"
            f"<model-viewer src='data:model/gltf-binary;base64,{b64}' "
            f"camera-controls auto-rotate shadow-intensity='1' "
            f"style='width:100%;height:320px;background:#1b1b1f;border-radius:8px'></model-viewer></div>")

    display(HTML(
        "<script type='module' "
        "src='https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js'></script>"
        "<style>.g{display:grid;grid-template-columns:repeat(auto-fit,minmax(330px,1fr));gap:14px;"
        "font-family:system-ui,sans-serif}.c{background:#111;color:#eee;padding:10px;border-radius:10px}"
        ".c span{float:right;opacity:.6;font-size:12px}</style>"
        f"<div class=g>{''.join(cards)}</div>"))

In [ ]:
# Zip everything and download, so you can open the GLBs in Blender / your Three.js viewer.
import shutil
from pathlib import Path
from google.colab import files

shutil.make_archive("/content/eval3d_results", "zip", "/content/eval3d/out")
size = Path("/content/eval3d_results.zip").stat().st_size / 1e6
print(f"eval3d_results.zip — {size:.1f} MB")
files.download("/content/eval3d_results.zip")

---
## What to look for when judging the output

For furniture / interior assets specifically:

- **Straight edges and flat planes.** Table tops, cabinet doors and chair legs are where these
  models fail most visibly — they tend to produce wobbly, organic geometry. This is usually the
  deciding factor.
- **The unseen back face.** Everything behind the camera is hallucinated. Rotate 180 degrees.
- **Texture legibility.** Wood grain and fabric weave usually survive; printed text and logos never do.
- **Poly count and topology** (`len(mesh.faces)`) — check it is sane for a web viewer.
- **Scale.** All three output unit-normalised meshes with no real-world dimensions. For interior
  design you will need to re-scale from a known reference, which is app-side work, not model work.

### Once you have a verdict
- **Good enough** -> next step is a RunPod serverless worker wrapping the winning model, called
  from your Spring Boot backend, writing GLBs to S3.
- **Not good enough** -> compare against a commercial API (Tripo, Meshy) before writing off the approach.